In [ ]:

cd /Users/cl1704/PycharmProjects/embedding_dimension/RNN


In [ ]:
# Modulate the curvature index of each neuron.
import torch
from torch import optim
import torch.nn as nn
import seaborn as sns
from matplotlib import gridspec
from Experiment_10.ParametricWMTask import generate_trials
import matplotlib.pyplot as plt
import matplotlib
import umap
from sklearn.decomposition import PCA
plt.rcParams["axes.grid"] = False
matplotlib.rcParams['axes.linewidth'] = .5
device= 'cpu'

In [ ]:
pwd

In [ ]:
class Net(torch.nn.Module):
    def __init__(self, n, alpha=.2, sigma_rec=0., sigma_in=0.2, input_size=6, output_size=2, dale=False,
                 lambda_std=0., base_in=0., beta=10):
        super(Net, self).__init__()
        self.alpha = torch.tensor(alpha)
        self.sigma_rec = torch.tensor(sigma_rec)
        self.sigma_in = torch.tensor(sigma_in)
        self.n = n
        self.input_size = input_size
        self.output_size = output_size
        self.dale = dale
        self.lambda_std = lambda_std
        self.base_in = base_in

        self.beta = beta
        self.activation = torch.nn.Softplus(beta=self.beta)
        # self.activation = torch.nn.ReLU()

        # Connectivity
        self.recurrent_layer = nn.Linear(self.n, self.n, bias=True)
        #self.recurrent_layer.weight.data.fill_diagonal_(0)

        self.recurrent_layer.bias.data.normal_(mean=0.2, std=0).to(device=device)
        self.recurrent_layer.bias.requires_grad = True

        self.input_layer = nn.Linear(self.input_size, self.n, bias=True)
        self.input_layer.bias.data.normal_(mean=self.base_in, std=0.).to(device=device)
        self.input_layer.bias.requires_grad = True
        # self.input_layer.weight.data = torch.where(torch.rand_like(self.input_layer.weight.data)> 0.5, -self.input_layer.weight.data, self.input_layer.weight.data)



    # Dynamics
    def forward(self, u, training=False):
        t = u.shape[1]
        states = torch.zeros(u.shape[0], 1, self.n, device=device)

        batch_size = states.shape[0]
        input_noise = torch.sqrt(2 / self.alpha * self.sigma_in ** 2) * torch.empty(batch_size, t,
                                                                                    self.input_size).normal_(mean=0,
                                                                                                             std=1).to(
            device=device)

        inputs = self.input_layer(u + input_noise)
        for i in range(t - 1):
            state_new = (1 - self.alpha) * states[:, i, :] + self.alpha * (
                self.activation(self.recurrent_layer(states[:, i, :]) + inputs[:, i, :] ))
            states = torch.cat((states, state_new.unsqueeze_(1)), 1)

        return states

In [ ]:
import torch

def compute_zscored_derivatives(y, dt=1.0, epsilon=1e-8):
    """
    1. Z-scores each neuron across trials and time.
    2. Calculates the temporal derivative (dot_z) for each neuron.

    Args:
        y: Tensor of shape (trials, time, neurons)
        dt: Time step between samples
        epsilon: Small constant to avoid division by zero in z-scoring

    Returns:
        z_scored_y: The standardized tensor
        dot_z: The temporal derivative of the standardized tensor
    """
    # 1. Z-Score Calculation
    # We calculate mean and std for each neuron (dim=2)
    # across all trials (dim=0) and timepoints (dim=1)
    mean = y.mean(dim=(0, 1), keepdim=True)
    std = y.std(dim=(0, 1), keepdim=True)

    # Standardize: z = (y - mu) / sigma
    z_scored_y = (y - mean) / (std + epsilon)

    # 2. Estimate Temporal Derivative (dot_z)
    # Using torch.gradient for second-order accurate central differences
    # in the interior and first-order at the boundaries.
    dot_z = torch.gradient(z_scored_y, spacing=dt, dim=1)[0]

    return z_scored_y, dot_z

# Example Usage:
# z_y, dz = compute_zscored_derivatives(y_tensor, dt=0.01)

In [ ]:
#Train high dimensional RNN
rnn = Net(n=250, input_size=2,
          sigma_in=0.,
          sigma_rec = 0.,
          base_in=0.1,
          lambda_std=0.4,
          beta=
          20,
          alpha = .05)
criterion = nn.MSELoss()
optimizer = optim.Adam(rnn.parameters(), lr=0.001,weight_decay = .001)
#scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5)

# 4. Training Loop
epochs = 1000
n_trials = 25
n_steps = 100
u, z, mask, conditions = generate_trials(n_trials=5,n_conditions =25,n_t = 200)

for epoch in range(epochs):
    rnn.train()
    optimizer.zero_grad()

    # Forward pass
    y = rnn(u)
    z_scored_y, dot_z = compute_zscored_derivatives(y, dt=1.0, epsilon=1e-8)
    g = torch.mean(torch.abs(dot_z),dim = (0,1))
    g_loss = torch.mean(g)
    reg_loss =  torch.std(torch.std(y, dim=[0, 1])) + torch.std(torch.mean(y, dim=[0, 1]))

   # y = (y-torch.mean(y,dim=(0,1)))/torch.std(y,dim=(0,1))

    # Dimensionality
    eigs = torch.sort(torch.real(
            torch.linalg.eigvals(torch.cov(z_scored_y.reshape(-1, z_scored_y.shape[2]).t()))),
                          descending=True)[0]
    fve = eigs/torch.sum(eigs)
    dim = 4
    var_1 = fve[dim-1]
    var_2 = torch.sum(fve[dim:])
    # Loss backprogation
    loss = 1*(var_1 - 1/dim)**2 + var_2 + (g_loss-1)**2 + .4*reg_loss
    #loss = slam_loss
    loss.backward()
    optimizer.step()
    rnn.recurrent_layer.bias.data = torch.nn.ReLU()(rnn.recurrent_layer.bias.data)

    # Print statements
    if epoch % 10 == 0:
        print(f"Epoch [{epoch}/{epochs}] | var_1: {var_1.item():.4f} | var_2: {var_2.item():.4f} | g: {g_loss.item():.4f} | std_c: {torch.std(g).item():.4f}")

In [ ]:
# Scree plot
x = rnn(u).detach()
x = (x-torch.mean(x,dim=(0,1)))/torch.std(x,dim=(0,1))
pca = PCA().fit(x.reshape(-1,x.shape[2]).detach())

fig = plt.figure(figsize=(1.75,1.75))
gs = gridspec.GridSpec(1,1)
ax=fig.add_subplot(gs[0,0])
ax.plot( pca.explained_variance_ratio_[:10],marker='.')
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)
ax.xaxis.set_tick_params(labelsize=7, bottom=True)
ax.yaxis.set_tick_params(labelsize=7, left=True)
ax.set_xlabel("PC",fontsize=8)
ax.set_ylabel("Variance (%)",fontsize=8)



In [ ]:
#########################################################################
################ Time series embedding for bwm data  ####################
#########################################################################
from sklearn.manifold import TSNE
fig = plt.figure(figsize=(8,3))
from scipy.spatial import ConvexHull
gs = gridspec.GridSpec(1,3,wspace = .3)
ax0=fig.add_subplot(gs[0,0])
ax1=fig.add_subplot(gs[0,1])
ax2=fig.add_subplot(gs[0,2])

u, z, mask, conditions = generate_trials(n_trials=1,n_conditions =25,n_t = 100)

x = rnn(u)
h = rnn.recurrent_layer(x) + rnn.input_layer(u)

x  = x.detach()
x = (x-torch.mean(x,dim=(0,1)))/torch.std(x,dim=(0,1))
#curvature_indices = compute_nonlinearity_index(x, window_size=5, dt=1.0)
#x_proj = TSNE(n_components = 2,perplexity=50).fit_transform(x.numpy().reshape(-1,x.shape[2]).T)
# fit = umap.UMAP( densmap=True,dens_lambda=2)
# x_proj= fit.fit_transform(x.reshape(-1,x.shape[2]).T)
x_proj = PCA().fit_transform(x.numpy().reshape(-1,x.shape[2]).T)
#x_proj = projection_model(torch.FloatTensor(data.T)).detach().numpy()


sns.scatterplot(x=x_proj[:,0], y=x_proj[:,1],s=4,ax=ax0,legend = False,lw = 0,alpha = .2,color='gray')

sns.scatterplot(x=x_proj[:,0], y=x_proj[:,1],s=6,ax=ax1,legend = False,lw = 0,alpha = .5,hue = g.detach(),palette = 'turbo')

for ax in [ax0,ax1]:
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)
    ax.spines['left'].set_visible(False)
    ax.spines['bottom'].set_visible(False)
    ax.axes.get_xaxis().set_visible(False)
    ax.axes.get_yaxis().set_visible(False)
    ax.set_aspect('equal')
# plt.savefig('Figures/Figure_1_b.jpg',
#             format='jpeg',
#             dpi=450,
#             bbox_inches='tight', transparent=True)


In [ ]:
curvature.shape

In [ ]:
plt.figure()

sns.histplot(x = torch.std(x,dim=(0,1)).detach().numpy())

In [ ]:
##########################################
########## Plotting manifolds ############
##########################################
u, z, mask, conditions = generate_trials(n_trials=1,n_conditions =1000,n_t =200)
rnn.alpha = torch.tensor(.05).float()
rnn.sigma_in =0
x = rnn(u,training=False)
z_scored_y, g = compute_zscored_derivatives(x, dt=1.0, epsilon=1e-8)
g = g.detach().numpy()

#x = (x - torch.mean(x,dim=(0,1)))/ torch.std(x,dim=(0,1))
x = x.detach().numpy()
pca = PCA().fit(x.reshape(-1,x.shape[2]))


rows = []
x = torch.tensor(x).float()
#net.recurrent_layer.weight.data = net.recurrent_layer.weight.data + .005*(2*torch.randn_like(net.recurrent_layer.weight.data)-1)
h = rnn.recurrent_layer(x)+rnn.input_layer(u)

flow_field = -x + rnn.activation(h)
e = torch.exp(rnn.beta * h)
b =rnn.beta* e/(1+e)**2
curvature =  (flow_field @ rnn.recurrent_layer.weight.data.t())**2 * b

x.shape

In [ ]:
import numpy as np
import pandas as pd
neuron = np.argsort(np.mean(g,axis = (0,1)))[-1]
for trial in range(x.shape[0]):
    trajectory = x[trial,:,:]
    proj_trajectory = pca.transform(trajectory)
    for t in range(x.shape[1]-1):

        rows.append({'t': t,
             'trial': trial,
             'x': proj_trajectory[t,0],
             'y': proj_trajectory[t,1],
             'z': proj_trajectory[t,2],
             'c': torch.linalg.norm(curvature[trial,t,neuron]).item(),
            'g': g[trial,t,neuron].item(),

              'b':torch.linalg.norm(b[trial,t,neuron]).item()})

new_df = pd.DataFrame(rows)


In [ ]:
curvature.shape

In [ ]:
fig = plt.figure(figsize=(4,4))
from scipy.spatial import ConvexHull
gs = gridspec.GridSpec(1,1)

ax0=fig.add_subplot(gs[0,0])

sns.scatterplot(data = new_df, x = 'z', y = 'y', hue = 'g',palette = 'turbo',s=6,legend=False,alpha = 1,ax=ax0)

for ax in [ax0]:
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)
    ax.spines['bottom'].set_visible(False)
    ax.spines['left'].set_visible(False)
    ax.axes.get_xaxis().set_visible(False)
    ax.axes.get_yaxis().set_visible(False)
    # plt.savefig('Figures/Figure_boundary_8.jpg',
    #         format='jpeg',
    #         dpi=450,
    #         bbox_inches='tight', transparent=True)
# ax.spines['left'].set_color('black')
# ax.spines['bottom'].set_color('black')


In [ ]:
import numpy as np

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px


fig = make_subplots(
     rows=1, cols=1,
     specs=[[{"type": "scene"}]])

sub_df = new_df
fig.add_trace(go.Scatter3d(x=sub_df.x.values,
                          y=sub_df.y.values,
                          z=sub_df.z.values,
                              mode='markers',

                        marker=dict(
                            color=sub_df.b.values,
                            colorscale='turbo',
                            opacity=1,
                            size = 2
                        )), row=1, col=1)
#px.colors.diverging.balance
#
fig.update_layout( scene = dict(xaxis = dict(showgrid = False,showticklabels = False),
                                   yaxis = dict(showgrid = False,showticklabels = False),
                                   zaxis = dict(showgrid = False,showticklabels = False),

             ))
fig.update_scenes(xaxis_visible=False, yaxis_visible=False,zaxis_visible=False )

In [ ]:
##########################################

x = rnn(u)
z_scored_y, dot_z = compute_zscored_derivatives(x, dt=1.0, epsilon=1e-8)
#g = torch.mean(torch.abs(dot_z),dim = (0,1))
#slam = calculate_discrete_intensity(x).detach()
#x = (x - torch.mean(x,dim=(0,1)))/ torch.std(x,dim=(0,1))
x = x.detach()
xdot=-x+rnn.activation(rnn.recurrent_layer(x)+rnn.input_layer(u))
xdot = xdot.detach()
trial = 6


#curvature = compute_instantaneous_curvature(x[trial,:,neuron], xdot[trial,:,neuron], dt=1)

rows=[]
for t in range(x.shape[1]-2):
    rows.append({'t':t,
                 'x':x[trial,t,neuron].item(),
                 'y':xdot[trial,t,neuron].item(),
                 'doty':dot_z[trial,t,neuron].item()})
df = pd.DataFrame(rows)
fig = plt.figure(figsize=(2,2))
from scipy.spatial import ConvexHull
gs = gridspec.GridSpec(1,1)

ax=fig.add_subplot(gs[0,0])
sns.scatterplot(data = df, x = 'x',y='y',hue='doty',legend=False,ax=ax,palette='turbo',lw=0)

a = -1  # Slope
b = np.log(2)/rnn.beta  # Intercept

# Create data
x = np.linspace(-.1, .1, 100)
y = a * x + b

# Plot
ax.plot(x, y, label=f'y = {a}x + {b}')


ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)
ax.spines['bottom'].set_visible(False)
ax.spines['left'].set_visible(False)
ax.axes.get_xaxis().set_visible(False)
ax.axes.get_yaxis().set_visible(False)


In [ ]:
g.shape

In [ ]:
np.argsort(max_curvature.detach())[-1]

In [ ]:
plt.figure()
sns.histplot(x=max_curvature.detach())

In [ ]:
import torch

def calculate_nonlinearity_map(y_tensor, W, b, beta=1.0):
    """
    Returns a tensor of shape (n_trial, n_time, n_neurons) containing
    the normalized non-linearity index for every neuron at every moment.
    """
    # 1. Compute pre-activation inputs
    # y_tensor shape: (n_trial, n_time, n_neurons)
    u = torch.matmul(y_tensor, W.t()) + b

    # 2. Compute the vector field (the 'flow')
    # f(u) = (1/beta) * log(1 + exp(beta * u))
    f_u = torch.nn.functional.softplus(u, beta=beta)
    F_y = -y_tensor + f_u

    # 3. Compute the change in input 'u' along the flow (direction of F_y)
    # AV represents the 'input velocity' for each neuron
    AV = torch.matmul(F_y, W.t())

    # 4. Derivatives for Softplus with Beta gain
    # f'(u) is the sigmoid (slope)
    # f''(u) is the curvature
    sigma_beta_u = torch.sigmoid(beta * u)
    f_double_prime = beta * sigma_beta_u * (1 - sigma_beta_u)

    # 5. Calculate Taylor Coefficients
    # Linear coefficient (L): the first-order rate of change
    # Note: F_y is the actual change, we check the linear prediction error
    L = torch.abs(-F_y + sigma_beta_u * AV)

    # Quadratic coefficient (E): the second-order curvature contribution
    E = 0.5 * torch.abs(f_double_prime * (AV**2))

    # 6. Compute Normalized Index
    # Shape: (n_trial, n_time, n_neurons)
    index_tensor = E / (L + 1e-12)

    return index_tensor

In [ ]:

x = rnn(u)
x = x.detach()
curvature = calculate_nonlinearity_map(x, rnn.recurrent_layer.weight.data, rnn.input_layer(u)[0,0,:], beta=rnn.beta)

In [ ]:
plt.figure()
for i in range(curvature.shape[0]):
    plt.plot(curvature[i,:,79].detach())

In [ ]:
torch.argsort(torch.mean(curvature,dim=(0,1)))[0]

In [ ]:
torch.mean(curvature,dim=(0,1))[81]